In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/bor7sbozhkov/captcha-msu/unlabelled.parquet
/kaggle/input/datasets/bor7sbozhkov/captcha-msu/train.parquet
/kaggle/input/datasets/bor7sbozhkov/captcha-msu/test.parquet


imports

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier, StackingClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.metrics import accuracy_score, roc_auc_score

import dataset

In [3]:
name_tst = '/kaggle/input/datasets/bor7sbozhkov/captcha-msu/test.parquet'
name_tr = '/kaggle/input/datasets/bor7sbozhkov/captcha-msu/train.parquet'
unlabelled = '/kaggle/input/datasets/bor7sbozhkov/captcha-msu/unlabelled.parquet'

data = pd.read_parquet(name_tr)
tst = pd.read_parquet(name_tst)
unlabel = pd.read_parquet(unlabelled)

data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   target                      1000 non-null   int64  
 1   relative_captcha_init_time  1000 non-null   float64
 2   mouse_events_total          1000 non-null   int64  
 3   mouse_events                1000 non-null   object 
 4   touch_events_total          1000 non-null   int64  
 5   touch_events                1000 non-null   object 
 6   pointerdown_timestamp       1000 non-null   float64
 7   pointerdown_x               1000 non-null   int64  
 8   pointerdown_y               1000 non-null   int64  
 9   pointerup_timestamp         1000 non-null   float64
 10  pointerup_x                 1000 non-null   int64  
 11  pointerup_y                 1000 non-null   int64  
 12  hover_timestamp             1000 non-null   float64
 13  hover_x                     1000 n

In [4]:
data['target'].value_counts()

target
1    500
0    500
Name: count, dtype: int64

Функция для работы с json

In [5]:
def extract_touch_features(row):
    feats = {}
    raw = row['touch_events']

    if not isinstance(raw, str) or raw == '[]':

        for col in ['touch_cnt', 'touch_duration']:
            feats[col] = 0
        for param in ['force', 'radius_ratio', 'angle']:
            for stat in ['mean', 'std', 'min', 'max']:
                feats[f'touch_{param}_{stat}'] = 0.0
        return feats

    try:
        events = json.loads(raw)
    except:
        # битая JSON-строка
        return feats

    if not events:
        return feats

    times = np.array([e['timestamp_'] for e in events], dtype=float)
    xs = np.array([e['x_'] for e in events], dtype=float)
    ys = np.array([e['y_'] for e in events], dtype=float)
    forces = np.array([e['force_'] for e in events], dtype=float)
    rx = np.array([e['radiusX_'] for e in events], dtype=float)
    ry = np.array([e['radiusY_'] for e in events], dtype=float)
    angles = np.array([e['rotationAngle_'] for e in events], dtype=float)

    feats['touch_force_mean'] = forces.mean()
    feats['touch_force_std'] = forces.std()
    feats['touch_force_min'] = forces.min()
    feats['touch_force_max'] = forces.max()
    if len(times) > 1:
        feats['touch_force_trend'] = np.polyfit(times, forces, 1)[0]  # наклон
    else:
        feats['touch_force_trend'] = 0.0

    ratio = rx / (ry + 1e-6)
    feats['touch_radius_ratio_mean'] = ratio.mean()
    feats['touch_radius_ratio_std'] = ratio.std()
    feats['touch_radius_ratio_min'] = ratio.min()
    feats['touch_radius_ratio_max'] = ratio.max()

    feats['touch_angle_mean'] = angles.mean()
    feats['touch_angle_std'] = angles.std()
    feats['touch_angle_min'] = angles.min()
    feats['touch_angle_max'] = angles.max()
    if len(angles) > 1:
        angle_diff = np.diff(angles)
        feats['touch_angle_diff_mean'] = angle_diff.mean()
        feats['touch_angle_diff_std'] = angle_diff.std()
    else:
        feats['touch_angle_diff_mean'] = 0.0
        feats['touch_angle_diff_std'] = 0.0

    # Движение:
    dx = np.diff(xs)
    dy = np.diff(ys)
    dt = np.diff(times)
    valid = dt > 0
    if valid.sum() > 0:
        dx = dx[valid]
        dy = dy[valid]
        dt = dt[valid]
        speeds = np.sqrt(dx**2 + dy**2) / dt
        feats['touch_speed_mean'] = speeds.mean()
        feats['touch_speed_std'] = speeds.std()
        feats['touch_speed_max'] = speeds.max()
        if len(speeds) > 1:
            acc = np.diff(speeds) / dt[1:]
            feats['touch_acc_mean'] = np.abs(acc).mean()
            feats['touch_acc_std'] = acc.std()
        else:
            feats['touch_acc_mean'] = 0.0
            feats['touch_acc_std'] = 0.0
        path_length = (speeds * dt).sum()
        straight_dist = np.sqrt((xs[-1]-xs[0])**2 + (ys[-1]-ys[0])**2)
        feats['touch_path_length'] = path_length
        feats['touch_straightness'] = straight_dist / (path_length + 1e-5)
    else:
        # одно событие или нет движения
        for k in ['speed_mean','speed_std','speed_max','acc_mean','acc_std','path_length','straightness']:
            feats[f'touch_{k}'] = 0.0

    feats['touch_cnt'] = len(events)
    feats['touch_duration'] = times[-1] - times[0] if len(times) > 0 else 0

    return feats

Обработка touch_events 

In [6]:
tst_touch = tst.apply(extract_touch_features, axis=1, result_type='expand')
data_touch = data.apply(extract_touch_features, axis=1, result_type='expand')
unlabel_touch = unlabel.apply(extract_touch_features, axis=1, result_type='expand')

Функция для mouse_events

In [7]:
def extract_mouse_features(row):
    feats = {}
    raw = row.get('mouse_events')

    if not isinstance(raw, str) or raw == '[]':
        for key in ['cnt', 'duration', 'dt_mean', 'dt_std', 'dt_min', 'dt_max', 'dt_lt20',
                    'v_mean', 'v_std', 'v_max', 'v_peak_ratio',
                    'a_mean', 'a_std',
                    'path_length', 'straight_dist', 'tortuosity',
                    'bbox_area', 'angle_mean', 'angle_std']:
            feats[f'mouse_{key}'] = 0.0
        feats['mouse_total_ratio'] = 0.0
        return feats

    try:
        events = json.loads(raw)
    except:
        return feats  

    if len(events) < 2:

        feats['mouse_cnt'] = len(events)
        feats['mouse_duration'] = 0
        feats['mouse_total_ratio'] = row.get('mouse_events_total', 0) / max(len(events), 1)

        return feats

    # Извлекаем массивы
    t = np.array([e['timestamp_'] for e in events], dtype=float)
    x = np.array([e['x_'] for e in events], dtype=float)
    y = np.array([e['y_'] for e in events], dtype=float)

    dt = np.diff(t)
    
    valid = dt > 0
    if valid.sum() == 0:

        feats['mouse_cnt'] = len(events)
        feats['mouse_duration'] = 0
        feats['mouse_total_ratio'] = row.get('mouse_events_total', 0) / max(len(events), 1)
        return feats

    dt = dt[valid]
    dx = np.diff(x)[valid]
    dy = np.diff(y)[valid]


    feats['mouse_dt_mean'] = dt.mean()
    feats['mouse_dt_std'] = dt.std()
    feats['mouse_dt_min'] = dt.min()
    feats['mouse_dt_max'] = dt.max()
    feats['mouse_dt_lt20'] = (dt < 20).mean()  # доля быстрых интервалов

    # скорость
    speeds = np.sqrt(dx**2 + dy**2) / dt
    feats['mouse_v_mean'] = speeds.mean()
    feats['mouse_v_std'] = speeds.std()
    feats['mouse_v_max'] = speeds.max()
    feats['mouse_v_peak_ratio'] = speeds.max() / (speeds.mean() + 1e-6)

    #  ускорение
    if len(speeds) > 1:
        acc = np.diff(speeds) / dt[1:]
        feats['mouse_a_mean'] = np.abs(acc).mean()
        feats['mouse_a_std'] = acc.std()
    else:
        feats['mouse_a_mean'] = 0.0
        feats['mouse_a_std'] = 0.0

    #  геометрия 
    path_length = np.sum(speeds * dt)
    straight_dist = np.sqrt((x[-1] - x[0])**2 + (y[-1] - y[0])**2)
    feats['mouse_path_length'] = path_length
    feats['mouse_straight_dist'] = straight_dist
    feats['mouse_tortuosity'] = path_length / (straight_dist + 1e-6)

    min_x, max_x = x.min(), x.max()
    min_y, max_y = y.min(), y.max()
    feats['mouse_bbox_area'] = (max_x - min_x) * (max_y - min_y)

    vectors = np.column_stack([dx, dy])  # массив (N,2)
    norms = np.linalg.norm(vectors, axis=1)
    angles = []
    for i in range(len(vectors)-1):
        if norms[i] > 0 and norms[i+1] > 0:
            cos = np.dot(vectors[i], vectors[i+1]) / (norms[i] * norms[i+1])
            cos = np.clip(cos, -1.0, 1.0)
            angles.append(np.arccos(cos))
    if angles:
        angles = np.array(angles)
        feats['mouse_angle_mean'] = angles.mean()
        feats['mouse_angle_std'] = angles.std()
    else:
        feats['mouse_angle_mean'] = 0.0
        feats['mouse_angle_std'] = 0.0

    #общее
    feats['mouse_cnt'] = len(events)
    feats['mouse_duration'] = t[-1] - t[0]
    feats['mouse_total_ratio'] = row.get('mouse_events_total', 0) / max(len(events), 1)

    return feats

Обработка mouse_events

In [8]:
tst_mouse = tst.apply(extract_mouse_features, axis=1, result_type='expand')
data_mouse = data.apply(extract_mouse_features, axis=1, result_type='expand')
unlabel_mouse = unlabel.apply(extract_mouse_features, axis=1, result_type='expand')

Получение полных данных 

In [9]:
tst = pd.concat([tst, tst_touch, tst_mouse], axis=1)
data = pd.concat([data, data_touch, data_mouse], axis=1)
unlabel = pd.concat([unlabel, unlabel_touch, unlabel_mouse], axis=1)

Удаление старых ненужных фич

In [10]:
tst = tst.drop(['mouse_events', 'touch_events'], axis=1)
data = data.drop(['mouse_events', 'touch_events'], axis=1)
unlabel = unlabel.drop(['mouse_events', 'touch_events'], axis=1)

Отделение таргета

In [11]:
X_tr = data.drop(['target'], axis=1)
y_tr = data['target']

Использование IsolationForest на unlabelled

In [12]:
isof = IsolationForest(n_estimators = 500, contamination=0.2, random_state=42)
data_isof = pd.concat([X_tr, unlabel], axis=0)
isof.fit(data_isof)
X_tr['anomality'] = isof.decision_function(X_tr)
tst['anomality'] = isof.decision_function(tst)

train-test-split

In [13]:
X_train, X_val, y_train, y_val = train_test_split(X_tr, y_tr, stratify=y_tr)

CatBoost

In [14]:
cb = CatBoostClassifier(
    random_seed=42,
    learning_rate=0.03,
    iterations=2000,
    depth=6,
    l2_leaf_reg=3,
    eval_metric='AUC',
    early_stopping_rounds=200,
    verbose=100
)

XGBClassifier

In [15]:
xgb = XGBClassifier(random_seed = 42,
                    n_estimators = 1000,
                    learning_rate = 0.05,
                    max_depth = 5,
                    eval_metric = 'auc')

LGBMC

In [16]:
lgb = LGBMClassifier(random_seed=42,
                        n_estimators=1000,
                        learning_rate = 0.05,
                        max_depth = 5,
                        num_leaves=30,
                        verbose=-1)

fitting

In [17]:
cb.fit(X_train, y_train, eval_set=(X_val, y_val))

0:	test: 0.8729600	best: 0.8729600 (0)	total: 56.6ms	remaining: 1m 53s
100:	test: 0.9134720	best: 0.9223680 (6)	total: 262ms	remaining: 4.93s
200:	test: 0.9178240	best: 0.9223680 (6)	total: 455ms	remaining: 4.08s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.922368
bestIteration = 6

Shrink model to first 7 iterations.


CatBoostClassifier(depth=6, early_stopping_rounds=200, eval_metric='AUC', iterations=2000, l2_leaf_reg=3, learning_rate=0.03, random_seed=42, verbose=100)

In [18]:

xgb.fit(X_train, y_train)
lgb.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:13:33] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "random_seed" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


LGBMClassifier(learning_rate=0.05, max_depth=5, n_estimators=1000,
               num_leaves=30, random_seed=42, verbose=-1)

probability:

In [19]:
p_cb = cb.predict_proba(X_val)[:, 1]

In [20]:
p_lgb = lgb.predict_proba(X_val)[:, 1]
p_xgb = xgb.predict_proba(X_val)[:, 1]

Searching for the best option in model weights:

In [21]:
w = [0.5, 0.25, 0.25]
p_m = 0
for w_cb in np.arange(0.0, 0.9, 0.05):
    w_rest = (1-w_cb)/2
    p_w = (w_cb*p_cb + w_rest*(p_lgb+p_xgb))
    p_auc = roc_auc_score(y_val, p_w, max_fpr=0.1)
    if p_auc > p_m:
        p_m = p_auc
        w = [w_cb, w_rest, w_rest]
print(f"Best weights (cat, lgb, xgb): {w}, p_AUC={p_m}")

Best weights (cat, lgb, xgb): [np.float64(0.8500000000000001), np.float64(0.07499999999999996), np.float64(0.07499999999999996)], p_AUC=0.7632000000000001


fit

In [22]:
cb.fit(X_tr, y_tr, eval_set=(X_val, y_val))
xgb.fit(X_tr, y_tr)
lgb.fit(X_tr, y_tr)

0:	test: 0.8647680	best: 0.8647680 (0)	total: 3.38ms	remaining: 6.77s
100:	test: 0.9717760	best: 0.9717760 (100)	total: 226ms	remaining: 4.24s
200:	test: 0.9898240	best: 0.9898240 (199)	total: 450ms	remaining: 4.02s
300:	test: 0.9986560	best: 0.9986560 (299)	total: 662ms	remaining: 3.74s
400:	test: 1.0000000	best: 1.0000000 (400)	total: 882ms	remaining: 3.52s
500:	test: 1.0000000	best: 1.0000000 (400)	total: 1.1s	remaining: 3.3s
600:	test: 1.0000000	best: 1.0000000 (400)	total: 1.3s	remaining: 3.03s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 1
bestIteration = 400

Shrink model to first 401 iterations.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [18:13:36] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "random_seed" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


LGBMClassifier(learning_rate=0.05, max_depth=5, n_estimators=1000,
               num_leaves=30, random_seed=42, verbose=-1)

predict

In [23]:
p_cb = cb.predict_proba(tst)[:, 1]
p_lgb = lgb.predict_proba(tst)[:, 1]
p_xgb = xgb.predict_proba(tst)[:, 1]

p = w[0]*p_cb+w[1]*p_lgb+w[2]*p_xgb


submit

In [24]:
subm = pd.DataFrame({
    'id': range(len(p)),
    'prediction': p
})

subm.to_parquet('submission.parquet', index=False)